# [STARTER] Exercise - Building an Agentic RAG System

In this exercise, you will build an Agentic RAG (Retrieval-Augmented Generation) system that 
combines the power of AI agents with traditional RAG pipelines. You'll create an agent that 
can decide when and how to retrieve information from different sources, including vector 
databases, web search, and other tools.


## Challenge

Your challenge is to create an Agentic RAG system that can:

- Build a RAG pipeline as a tool that can be used by the agent
- Create an agent that can decide which tool to use based on the query
- Handle different types of queries intelligently
- Combine information from multiple sources when needed


## Setup
First, let's import the necessary libraries:

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
from typing import List
from dotenv import load_dotenv

from lib.agents import Agent
from lib.llm import LLM
from lib.state_machine import Run
from lib.messages import BaseMessage
from lib.tooling import tool
from lib.vector_db import VectorStoreManager, CorpusLoaderService
from lib.rag import RAG

In [3]:
load_dotenv()

True

In [4]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

## Load data to Vector DB

In [5]:
db = VectorStoreManager(OPENAI_API_KEY)
db

VectorStoreManager():<chromadb.api.client.Client object at 0x10f4592b0>

In [6]:
loader_service = CorpusLoaderService(db)

In [7]:
rag_llm = LLM(
    model="gpt-5-nano",
    temperature=0.3,
)

In [8]:
# TODO: Add the games pdf file path with the extenstion .pdf
# And define a store name in load_pdf() method

games_market_rag = RAG(
    llm=rag_llm,
    vector_store = loader_service.load_pdf("games_market","pdf/TheGamingIndustry2024.pdf")
)

VectorStore `games_market` ready!
Pages from `pdf/TheGamingIndustry2024.pdf` added!


In [9]:
result:Run = games_market_rag.invoke(
    "What's the  state of virtual reality"
)
print(result.get_final_state()["answer"])

[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
Based on the provided context, virtual reality (VR) is a major, expanding trend in gaming in 2024. It’s described as a technology that lets players immerse themselves in realistic, interactive digital worlds with high-fidelity visuals, intuitive controls, and immersive audio. The example Half-Life: Alyx is used to illustrate how VR can create visceral, believable environments where decisions have real consequences. VR is presented alongside AR, the Metaverse, and Web3 as key directions shaping the future of gaming, emphasizing immersive, sensory-rich experiences.


In [10]:
# TODO: Add the electric vehicles pdf file path with the extenstion .pdf
# And define a store name in load_pdf() method

electric_vehicles_rag = RAG(
    llm=rag_llm,
    vector_store = loader_service.load_pdf("electric_vehicles","pdf/GlobalEVOutlook2025.pdf")
)

VectorStore `electric_vehicles` ready!
Pages from `pdf/GlobalEVOutlook2025.pdf` added!


In [11]:
result:Run = electric_vehicles_rag.invoke("What was the number of electric car sales and their market share in Brazil in 2024?")
print(result.get_final_state()["answer"])

[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
About 125,000 electric car sales in Brazil in 2024, with an electric car market share of 6.5%.


## Tools

In a simple form, Agentic RAG can act like a router, choosing between multiple external sources to retrieve relevant information. These sources aren't limited to databases, they can also include tools like web search or APIs for services such as Slack or email.

In this case it will choose between two collections.

In [12]:
# TODO: Define a tool that returns result.get_final_state()["answer"]
# DONOT Forget about defining the tool docstrings
@tool
def search_global_ev_collection(query):
    """
    Searches the global electric vehicles collection for the given query.

    Args:
        query (str): The query string to search for.

    Returns:
        str: The answer from the electric vehicles RAG model.
    """
    result:Run = electric_vehicles_rag.invoke(query)
    return result.get_final_state()["answer"]

In [13]:
# TODO: Define a tool that returns result.get_final_state()["answer"]
# DONOT Forget about defining the tool docstrings
@tool
def search_games_market_report_collection(query):
    """Searches the games market collection for the given query.
    Args:
        query (str): The query string to search for.
    Returns:
        str: The answer from the games market RAG model.
    """
    result:Run = games_market_rag.invoke(query)
    return result.get_final_state()["answer"]

In [14]:
# TODO: Add the tools you have defined and the instructions to your agent
agentic_rag = Agent(
    model_name="gpt-5-nano",
    tools=[search_global_ev_collection, search_games_market_report_collection],    
    instructions="""You are an agent that can search the global electric vehicles collection and
                    the games market collection to provide answers to user queries."""
)

In [15]:
def print_messages(messages: List[BaseMessage]):
    for m in messages: 
        print(f" -> (role = {m.role}, content = {m.content}, tool_calls = {getattr(m, 'tool_calls', None)})")

## Run

In [16]:
run_1 = agentic_rag.invoke(
    query="Who won the 2025 Oscar for International Movie?", 
    session_id="oscar",
)

print("\nMessages from run 1:")
messages = run_1.get_final_state()["messages"]
print_messages(messages)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: web_search
[StateMachine] Executing step: comparison
[StateMachine] Terminating: __termination__

Messages from run 1:
 -> (role = system, content = You are an agent that can search the global electric vehicles collection and
                    the games market collection to provide answers to user queries., tool_calls = None)
 -> (role = user, content = W

In [17]:
run_2 = agentic_rag.invoke(
    query= (
        "Which two countries accounted for most of the electric car exports from " 
        "the Asia Pacific region (excluding China) in 2024?"
    ),
    session_id="electric_car",
)

print("\nMessages from run 2:")
messages = run_2.get_final_state()["messages"]
print_messages(messages)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: web_search
[StateMachine] Executing step: comparison
[StateMachine] Terminating: __termination__

Messages from run 2:
 -> (role = system, content = You are an agent that can search the global electric vehicles collection and
                    the games market collection to provide answers to user queries., tool_calls = None)
 -> (role = user, content = Which two countries accounted for most of the electric car exports from the Asia Pacific region (excluding China) in 2024?, tool_calls = None)
 -> (role = assistant, content = None, tool_calls = [Cha

In [18]:
run_3 = agentic_rag.invoke(
    query= (
        "Why is generative AI seen more as an accelerator than a replacement in game development?"
    ),
    session_id="games",
)

print("\nMessages from run 3:")
messages = run_3.get_final_state()["messages"]
print_messages(messages)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: web_search
[StateMachine] Executing step: comparison
[StateMachine] Terminating: __termination__

Messages from run 3:
 -> (role = system, content = You are an agent that can search the global electric vehicles collection and
                    the games market collection to provide answers to user queries., tool_calls = None)
 -> (role = user, content = Why is generative AI seen more as an accelerator than a replacement in game development?, tool_calls = None)
 -> (role = assistant, content = Generative AI is seen mainly as an accelerator in game development because it helps humans work faster, not replace them. Here are the core reasons:

- Speed and scale of ideation
  - It can generate many concepts, variants, and drafts (art, dialogue, level ideas) in moments, speeding up early exploration and iteration.
- Automation of repetit